<a href="https://colab.research.google.com/github/Metropoliya/AI/blob/main/Voice%20Book%20v%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q edge-tts PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 70.1 MB/s eta 0:00:00


In [8]:
import fitz
import re
import edge_tts
import asyncio
import os
from google.colab import drive

# --- Настройки ---
PDF_PATH = "new 666.pdf"
VOICE = "ru-RU-SvetlanaNeural"

# 1. Подключаем Google Диск (файлы будут сохраняться напрямую туда)
drive.mount('/content/drive')
# Создаем папку для аудиокниги на твоем Диске
SAVE_DIR = '/content/drive/MyDrive/Hackers_Audiobook'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Все файлы будут надежно сохраняться в: {SAVE_DIR}")

def get_clean_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    clean_text = re.sub(r'-\n', '', full_text)
    clean_text = re.sub(r'(?<!\n)\n(?!\n)', ' ', clean_text)
    return re.sub(r'\s+', ' ', clean_text).strip()

def chunk_text(text, max_length=2000):
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks, current_chunk = [], ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) < max_length:
            current_chunk += sentence + " "
        else:
            if current_chunk: chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk: chunks.append(current_chunk.strip())
    return chunks

async def generate_bulletproof_audio(text_chunks):
    print(f"Всего фрагментов: {len(text_chunks)}.")

    for i, chunk in enumerate(text_chunks):
        if not chunk.strip(): continue

        # Формируем имя файла с нулями (chunk_000.mp3, chunk_001.mp3)
        file_path = f"{SAVE_DIR}/chunk_{i:03d}.mp3"

        # ЛОГИКА ВОЗОБНОВЛЕНИЯ: Если файл уже есть на Диске — пропускаем!
        if os.path.exists(file_path):
            if i % 10 == 0:
                print(f"Фрагмент {i} уже существует на Диске. Пропускаем...")
            continue

        retries = 3
        while retries > 0:
            try:
                communicate = edge_tts.Communicate(chunk, VOICE)
                await communicate.save(file_path)

                if (i + 1) % 5 == 0:
                    print(f"Успешно сгенерирован и сохранен на Диск фрагмент {i + 1} / {len(text_chunks)}")

                await asyncio.sleep(1.5) # Вежливая пауза
                break # Выходим из цикла попыток при успехе

            except Exception as e:
                print(f"Сбой сети на куске {i}. Осталось попыток: {retries-1}. Ошибка: {e}")
                retries -= 1
                await asyncio.sleep(3)

    print("\n--- ВСЕ ФРАГМЕНТЫ УСПЕШНО СОХРАНЕНЫ НА GOOGLE ДИСК! ---")

# Запуск
text = get_clean_text_from_pdf(PDF_PATH)
chunks = chunk_text(text)
await generate_bulletproof_audio(chunks)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Все файлы будут надежно сохраняться в: /content/drive/MyDrive/Hackers_Audiobook
Всего фрагментов: 479.
Фрагмент 0 уже существует на Диске. Пропускаем...
Фрагмент 10 уже существует на Диске. Пропускаем...
Фрагмент 20 уже существует на Диске. Пропускаем...
Фрагмент 30 уже существует на Диске. Пропускаем...
Фрагмент 40 уже существует на Диске. Пропускаем...
Фрагмент 50 уже существует на Диске. Пропускаем...
Фрагмент 60 уже существует на Диске. Пропускаем...
Фрагмент 70 уже существует на Диске. Пропускаем...
Фрагмент 80 уже существует на Диске. Пропускаем...
Фрагмент 90 уже существует на Диске. Пропускаем...
Фрагмент 100 уже существует на Диске. Пропускаем...
Фрагмент 110 уже существует на Диске. Пропускаем...
Фрагмент 120 уже существует на Диске. Пропускаем...
Фрагмент 130 уже существует на Диске. Пропускаем...
Фрагмент 140 уже существует на Диске. Пропускаем...

In [9]:
import os
from google.colab import files

# Пути к папкам на твоем Диске
SAVE_DIR = '/content/drive/MyDrive/Hackers_Audiobook'
FINAL_AUDIO = '/content/drive/MyDrive/Hackers_Audiobook_FULL.mp3'

print("Собираем список файлов...")

# Находим все mp3 чанки и сортируем их строго по порядку
chunks = [f for f in os.listdir(SAVE_DIR) if f.startswith("chunk_") and f.endswith(".mp3")]
chunks.sort()

if not chunks:
    print("Файлы не найдены! Убедись, что генерация закончилась.")
else:
    print(f"Найдено {len(chunks)} фрагментов. Подготавливаем магию склейки...")

    # Создаем технический текстовый файл со списком для утилиты ffmpeg
    list_file_path = os.path.join(SAVE_DIR, "file_list.txt")
    with open(list_file_path, "w") as f:
        for chunk in chunks:
            f.write(f"file '{os.path.join(SAVE_DIR, chunk)}'\n")

    print("Склеиваем файлы воедино...")

    # Команда ffmpeg для склейки (-c copy означает мгновенное объединение без потери качества)
    os.system(f"ffmpeg -f concat -safe 0 -i '{list_file_path}' -c copy '{FINAL_AUDIO}' -y")

    print(f"\n--- ГОТОВО! Единая аудиокнига собрана и лежит тут: {FINAL_AUDIO} ---")

    print("Начинаю скачивание финальной книги на твой компьютер...")
    files.download(FINAL_AUDIO)

Собираем список файлов...
Найдено 479 фрагментов. Подготавливаем магию склейки...
Склеиваем файлы воедино...

--- ГОТОВО! Единая аудиокнига собрана и лежит тут: /content/drive/MyDrive/Hackers_Audiobook_FULL.mp3 ---
Начинаю скачивание финальной книги на твой компьютер...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>